# 🧠 GlioRad: Glioma Radiogenomics Brain MRI Segmentation — Kaggle Training Baseline

This self-contained notebook explores the multi-modal TCGA-LGG brain MRI dataset, trains the official baseline **U-Net** model on Kaggle using a GPU accelerator, evaluates segmentation metrics, and encrypts the submission.

### Workflow Steps:
1. **Environment & Hardware Setup**: Enable GPU (T4 / P100).
2. **Dataset Setup & Zero-Leakage Split**: Stratify by Patient ID (70% Train, 15% Val, 15% Test).
3. **Exploratory Data Analysis (EDA) & MRI Visualization**: Inspect Pre-contrast, FLAIR, and Post-contrast channels, tumor mask overlays, class balance, and axial tumor progression.
4. **Train U-Net**: Train with composite **Dice + BCE Loss**, tracking both **Training Loss**, **Validation Loss**, and **Validation Dice**.
5. **Evaluate & Visualize Predictions**: Compare Ground Truth vs. Predicted glioma segmentation overlays.
6. **Export & Encrypt Submission**: Generate Run-Length Encoded (RLE) CSV and encrypt via the official RSA `public_key.pem`.

## 1. Imports and GPU Verification

In [ ]:
!pip install -q rsa

import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import rsa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {device}")
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. Go to Settings -> Accelerator -> GPU T4 to enable GPU!")


## 2. Locate Kaggle Dataset & Perform Zero-Leakage Patient Split

In [ ]:
possible_paths = [
    Path("/kaggle/input/lgg-mri-segmentation/kaggle_3m"),
    Path("/kaggle/input/lgg-mri-segmentation"),
    Path("/kaggle/input/kaggle-3m/kaggle_3m"),
    Path("/kaggle/input/brain-mri-segmentation/kaggle_3m"),
    Path("/kaggle/input/brain-mri-segmentation")
]

data_root = None
for p in possible_paths:
    if p.exists() and len(list(p.glob("TCGA_*"))) > 0:
        data_root = p
        break

if data_root is None:
    raise FileNotFoundError(
        "❌ ERROR: Official TCGA-LGG Dataset not found in /kaggle/input!\n\n"
        "Please attach the official real dataset to this notebook:\n"
        "1. In the right-hand panel of your Kaggle notebook under 'Input', click '+ Add Input'.\n"
        "2. Search for: mateuszbuda/lgg-mri-segmentation\n"
        "3. Direct link: https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation\n"
        "4. Click '+' to mount it (contains 110 real patient folders and 3,929 real slices).\n"
        "5. Re-run this cell to begin training on the real clinical dataset!"
    )

print(f"✅ Found mounted TCGA-LGG dataset at: {data_root}")

patient_dirs = sorted([d for d in data_root.iterdir() if d.is_dir() and d.name.startswith("TCGA_")])
print(f"Discovered {len(patient_dirs)} patients.")

records = []
for p_dir in patient_dirs:
    patient_id = p_dir.name
    all_tifs = sorted(list(p_dir.glob("*.tif")))
    mask_files = {m.name: m for m in all_tifs if "_mask" in m.name}
    img_files = [img for img in all_tifs if "_mask" not in img.name]
    
    for img_path in img_files:
        exp_mask_name = f"{img_path.stem}_mask.tif"
        mask_path = mask_files.get(exp_mask_name)
        slice_num = int(img_path.stem.split("_")[-1]) if img_path.stem.split("_")[-1].isdigit() else 0
        has_tumor = 0
        if mask_path and mask_path.exists():
            has_tumor = int(np.array(Image.open(mask_path)).max() > 0)
        
        records.append({
            "patient_id": patient_id,
            "slice_idx": slice_num,
            "image_path": str(img_path),
            "mask_path": str(mask_path) if mask_path else None,
            "has_tumor": has_tumor
        })

df = pd.DataFrame(records)
print(f"Total slice pairs: {len(df)} (Tumor-positive slices: {df['has_tumor'].sum()})")

unique_patients = df["patient_id"].unique()
train_p, temp_p = train_test_split(unique_patients, test_size=0.30, random_state=42)
val_p, test_p = train_test_split(temp_p, test_size=0.50, random_state=42)

split_map = {**{p: "train" for p in train_p}, **{p: "val" for p in val_p}, **{p: "test" for p in test_p}}
df["split"] = df["patient_id"].map(split_map)

print(f"Train: {len(train_p)} patients ({len(df[df['split']=='train'])} slices)")
print(f"Val:   {len(val_p)} patients ({len(df[df['split']=='val'])} slices)")
print(f"Test:  {len(test_p)} patients ({len(df[df['split']=='test'])} slices)")


## 3. Exploratory Data Analysis (EDA) & Multi-Modal MRI Visualization

Each TCGA-LGG image is a 3-channel RGB composite representing three MRI sequences:
- **Channel 0 (Red)**: Pre-contrast T1
- **Channel 1 (Green)**: FLAIR (Fluid-Attenuated Inversion Recovery — primary sequence for LGG hyperintensity)
- **Channel 2 (Blue)**: Post-contrast T1c

Below we visualize the individual channels, ground-truth glioma mask overlay, slice-level class balance, and axial tumor probability.

In [ ]:
# Select a tumor-positive training slice for multi-channel visualization
pos_df = df[(df["split"] == "train") & (df["has_tumor"] == 1)]
sample_row = pos_df.iloc[len(pos_df) // 2] if len(pos_df) > 0 else df.iloc[0]

sample_rgb = np.array(Image.open(sample_row["image_path"]).convert("RGB"))
sample_mask = (np.array(Image.open(sample_row["mask_path"]).convert("L")) > 127).astype(np.uint8)

fig = plt.figure(figsize=(18, 10))
fig.suptitle(f"TCGA-LGG Multi-Modal Brain MRI & EDA — Patient {sample_row['patient_id']} (Slice #{sample_row['slice_idx']})",
             fontsize=15, fontweight="bold", y=0.98)

# 1. Pre-contrast Channel
ax1 = fig.add_subplot(2, 4, 1)
ax1.imshow(sample_rgb[:, :, 0], cmap="bone")
ax1.set_title("Channel 0: Pre-Contrast", fontweight="bold")
ax1.axis("off")

# 2. FLAIR Channel
ax2 = fig.add_subplot(2, 4, 2)
ax2.imshow(sample_rgb[:, :, 1], cmap="bone")
ax2.set_title("Channel 1: FLAIR Sequence", fontweight="bold")
ax2.axis("off")

# 3. Post-contrast Channel
ax3 = fig.add_subplot(2, 4, 3)
ax3.imshow(sample_rgb[:, :, 2], cmap="bone")
ax3.set_title("Channel 2: Post-Contrast", fontweight="bold")
ax3.axis("off")

# 4. RGB Composite + Ground Truth Tumor Overlay & Contour
ax4 = fig.add_subplot(2, 4, 4)
overlay_rgb = sample_rgb.astype(np.float32).copy()
fg = sample_mask > 0
overlay_rgb[fg, 0] = 0.55 * overlay_rgb[fg, 0]
overlay_rgb[fg, 1] = 0.55 * overlay_rgb[fg, 1] + 0.45 * 255
overlay_rgb[fg, 2] = 0.55 * overlay_rgb[fg, 2] + 0.45 * 210
ax4.imshow(np.clip(overlay_rgb, 0, 255).astype(np.uint8))
if sample_mask.sum() > 0:
    ax4.contour(sample_mask, levels=[0.5], colors=["#FF3366"], linewidths=2.0)
ax4.set_title(f"Glioma Mask Overlay ({sample_mask.sum():,} px²)", fontweight="bold")
ax4.axis("off")

# 5. Slice Positivity Class Balance
ax5 = fig.add_subplot(2, 2, 3)
counts = df["has_tumor"].value_counts().reindex([0, 1], fill_value=0)
bars = ax5.bar(["Normal Slice (0)", "Glioma Present (1)"], counts.values, color=["#4A90E2", "#E65540"], edgecolor="black", width=0.5)
ax5.set_title("Slice-Level Class Distribution (2D Axial Slices)", fontweight="bold")
ax5.set_ylabel("Number of Slices")
ax5.grid(axis="y", linestyle="--", alpha=0.4)

# 6. Axial Tumor Probability by Slice Index
ax6 = fig.add_subplot(2, 2, 4)
axial_freq = df.groupby("slice_idx")["has_tumor"].mean() * 100.0
ax6.plot(axial_freq.index, axial_freq.values, color="#8E44AD", linewidth=2.2)
ax6.fill_between(axial_freq.index, axial_freq.values, color="#8E44AD", alpha=0.2)
ax6.set_title("Tumor Frequency Across Axial Slice Position", fontweight="bold")
ax6.set_xlabel("Axial Slice Index (Inferior → Superior)")
ax6.set_ylabel("Tumor Presence (%)")
ax6.grid(linestyle="--", alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


## 4. PyTorch Dataset, U-Net Architecture & Dice+BCE Loss

In [ ]:
class KaggleLGGDataset(Dataset):
    def __init__(self, slice_df, is_test=False):
        self.df = slice_df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        img_np = np.array(img, dtype=np.float32) / 255.0
        img_t = torch.from_numpy(img_np).permute(2, 0, 1)

        mask_t = torch.zeros((1, img_t.shape[1], img_t.shape[2]), dtype=torch.float32)
        if not self.is_test and row["mask_path"]:
            mask = Image.open(row["mask_path"]).convert("L")
            bin_mask = (np.array(mask, dtype=np.float32) > 127.0).astype(np.float32)
            mask_t = torch.from_numpy(bin_mask).unsqueeze(0)

        return img_t, mask_t, row["patient_id"], row["slice_idx"]

train_ds = KaggleLGGDataset(df[df["split"] == "train"])
val_ds = KaggleLGGDataset(df[df["split"] == "val"])
test_ds = KaggleLGGDataset(df[df["split"] == "test"], is_test=True)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_c=3, out_c=1, f=32):
        super().__init__()
        self.inc = DoubleConv(in_c, f)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f, f*2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f*2, f*4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f*4, f*8))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(f*8, f*16))
        self.drop = nn.Dropout2d(0.2)
        
        self.up1 = nn.ConvTranspose2d(f*16, f*8, 2, stride=2)
        self.conv1 = DoubleConv(f*16, f*8)
        self.up2 = nn.ConvTranspose2d(f*8, f*4, 2, stride=2)
        self.conv2 = DoubleConv(f*8, f*4)
        self.up3 = nn.ConvTranspose2d(f*4, f*2, 2, stride=2)
        self.conv3 = DoubleConv(f*4, f*2)
        self.up4 = nn.ConvTranspose2d(f*2, f, 2, stride=2)
        self.conv4 = DoubleConv(f*2, f)
        self.outc = nn.Conv2d(f, out_c, 1)
        
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.drop(self.down4(x4))
        
        x = self.conv1(torch.cat([x4, self.up1(x5)], dim=1))
        x = self.conv2(torch.cat([x3, self.up2(x)], dim=1))
        x = self.conv3(torch.cat([x2, self.up3(x)], dim=1))
        x = self.conv4(torch.cat([x1, self.up4(x)], dim=1))
        return self.outc(x)

class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.6, bce_weight=0.4):
        super().__init__()
        self.dw = dice_weight
        self.bw = bce_weight
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits).view(logits.size(0), -1)
        targs = targets.view(targets.size(0), -1)
        intersection = (probs * targs).sum(dim=1)
        dice = (2.0 * intersection + 1e-5) / (probs.sum(dim=1) + targs.sum(dim=1) + 1e-5)
        dice_loss = 1.0 - dice.mean()
        return (self.bw * bce_loss) + (self.dw * dice_loss)

model = UNet(in_c=3, out_c=1, f=32).to(device)
criterion = DiceBCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
print("Model and loss initialized!")


## 5. Training Loop on GPU (Tracking Train Loss, Val Loss & Val Dice)

In [ ]:
def evaluate_val(model, loader, criterion):
    model.eval()
    dices = []
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks, _, _ in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            logits = model(imgs)
            loss = criterion(logits, masks)
            val_loss += loss.item()
            
            preds = (torch.sigmoid(logits) > 0.5).float()
            p_flat = preds.view(preds.size(0), -1)
            m_flat = masks.view(masks.size(0), -1)
            
            for i in range(p_flat.size(0)):
                inter = (p_flat[i] * m_flat[i]).sum().item()
                total = p_flat[i].sum().item() + m_flat[i].sum().item()
                dice = 1.0 if total == 0 else (2.0 * inter / total)
                dices.append(dice)
    return val_loss / max(1, len(loader)), float(np.mean(dices))

epochs = 8
best_dice = 0.0
history = {"train_loss": [], "val_loss": [], "val_dice": []}

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
    for imgs, masks, _, _ in pbar:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({"train_loss": f"{loss.item():.4f}"})
        
    avg_train_loss = train_loss / max(1, len(train_loader))
    avg_val_loss, val_dice = evaluate_val(model, val_loader, criterion)
    scheduler.step(val_dice)
    
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_dice"].append(val_dice)
    
    print(f"Epoch [{epoch:02d}/{epochs:02d}] -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Mean Slice Dice: {val_dice:.4f}")
    
    if val_dice >= best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "/kaggle/working/best_unet_model.pt")

print(f"\n✅ Training complete! Best Validation Mean Slice Dice: {best_dice:.4f}")


## 5.5 Visual Verification: Ground Truth vs. Predicted Glioma Segmentation

In [ ]:
# Find a tumor-positive validation sample to visualize predictions
val_sample_idx = 0
for i in range(len(val_ds)):
    _, m_t, _, _ = val_ds[i]
    if m_t.sum() > 0:
        val_sample_idx = i
        break

v_img, v_mask, v_pid, v_sidx = val_ds[val_sample_idx]
model.eval()
with torch.no_grad():
    v_pred = (torch.sigmoid(model(v_img.unsqueeze(0).to(device))) > 0.5).float().cpu().squeeze().numpy()

v_img_np = v_img.permute(1, 2, 0).numpy()
v_gt_np = v_mask.squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(v_img_np)
axes[0].set_title(f"MRI Composite ({v_pid} #{v_sidx})", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(v_img_np)
if v_gt_np.sum() > 0:
    axes[1].contour(v_gt_np, levels=[0.5], colors=["#00FFCC"], linewidths=2)
axes[1].imshow(v_gt_np, cmap="cool", alpha=0.45 * (v_gt_np > 0))
axes[1].set_title("Ground Truth FLAIR Abnormality", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(v_img_np)
if v_pred.sum() > 0:
    axes[2].contour(v_pred, levels=[0.5], colors=["#FF3366"], linewidths=2)
axes[2].imshow(v_pred, cmap="autumn", alpha=0.45 * (v_pred > 0))
axes[2].set_title("U-Net Predicted Tumor Segmentation", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()


## 6. Run-Length Encoding (RLE) & Test Prediction Generation

In [ ]:
def mask_to_rle(binary_mask):
    """Converts a 2D binary numpy mask (256, 256) to Run-Length Encoded string."""
    dots = binary_mask.flatten()
    runs = np.where(dots[1:] != dots[:-1])[0] + 2
    runs = np.concatenate([[1] if dots[0] else [], runs, [len(dots) + 1] if dots[-1] else []])
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

model.load_state_dict(torch.load("/kaggle/working/best_unet_model.pt"))
model.eval()

submission_rows = []
with torch.no_grad():
    for imgs, _, p_ids, s_idxs in tqdm(test_loader, desc="Predicting Test Set"):
        imgs = imgs.to(device)
        logits = model(imgs)
        preds = (torch.sigmoid(logits) > 0.5).cpu().numpy().squeeze(1)
        
        for i in range(len(preds)):
            mask = preds[i].astype(np.uint8)
            rle_str = mask_to_rle(mask) if mask.sum() > 0 else ""
            submission_rows.append({
                "patient_id": p_ids[i],
                "slice_idx": int(s_idxs[i]),
                "rle_mask": rle_str
            })

sub_df = pd.DataFrame(submission_rows)
raw_csv_path = "/kaggle/working/my_team_submission.csv"
sub_df.to_csv(raw_csv_path, index=False)
print(f"Exported {len(sub_df)} prediction rows to: {raw_csv_path}")
sub_df.head(10)


## 7. Official RSA-2048 Encryption for GitHub PR
Encrypts `my_team_submission.csv` into `my_team_submission.csv.enc` using the embedded challenge public key:

In [ ]:
PUBLIC_KEY_PEM = """-----BEGIN RSA PUBLIC KEY-----
MIIBCgKCAQEAuukYzCRwJ9NpBG9eU3Mhd/zlw2oPYCavrIHBiQ5W11xxxWjWpKNf
QA3ofeNKFCn8jUgemV8aZALoS3TQqaf512Fule0EpR9L92QyS4uZhQYjJxWyqhOK
msfPBpMyCv6TH/3TtEaCxKgWXMDfWD8YWBbIu9/ksKSy6Cv/Fb8pRnkOXd8QN6aA
l3HNUjbTAqogJcm+05UrzgmNiU6ci8QXYSbwpGO6glQS67242R+ru9mJ5TfLyy3e
8NFdm7LubfaSZLrPOSqDAOzhylESfXuI/F261nYxWiFkSlE+t+ZOq+/62ZhP1d+J
470eoQDCu52QYIv+8epZoP2114q8sec9swIDAQAB
-----END RSA PUBLIC KEY-----"""

def encrypt_submission(csv_path, pem_string):
    public_key = rsa.PublicKey.load_pkcs1(pem_string.strip().encode('utf-8'))
    with open(csv_path, 'rb') as f:
        file_data = f.read()
        
    encrypted_data = b""
    chunk_size = 245
    for i in range(0, len(file_data), chunk_size):
        encrypted_data += rsa.encrypt(file_data[i:i + chunk_size], public_key)
        
    enc_path = csv_path + ".enc"
    with open(enc_path, 'wb') as f:
        f.write(encrypted_data)
    print(f"✅ SUCCESS! Encrypted file created at: {enc_path}")
    print(f"File size: {len(encrypted_data):,} bytes.")

encrypt_submission(raw_csv_path, PUBLIC_KEY_PEM)
